# Re-scoring real model predictions on Parse (Rhaister + simple baselines)

**Goal:** check whether the ranking of prediction methods on the Parse donor split changes when the evaluation removes the shared PBS shift.

**Steps**
1. Run Rhaister (Svensson et al. 2026) on its own Parse split (`parse/fewshot_donor_split_0`: query donors 1, 4, 9, 12) with `HP_SAVE_PREDICTIONS=1`, which saves expression-delta truths and predictions for the test rows. Training-row truths are read from Rhaister's own data cache (`D_train.npy` + `meta.json`).
2. From the saved truths, build the simple predictors: **additive** (training-donor mean + query-context panel offset), **context mean** (mean of the query context's observed panel deltas; no cytokine information) and **inert shift** (mean delta of inert cytokines in training donors; no cytokine or donor information).
3. Score all predictors with `pearson_delta` (as published, deltas against PBS) and after **inert re-referencing**: in every query context, the mean over the inert test cytokines is subtracted from truth and from each predictor's own predictions, so the shared PBS shift is removed on both sides. Scores are reported for immune test cytokines.

In [1]:
import os, re, subprocess
import numpy as np, pandas as pd
RH_DIR = "Rhaister"
SPLIT = "parse/fewshot_donor_split_0"
EXP = "rescoring"
RESULTS_DIR = "results/rhaister_rescoring_strict"
os.makedirs(RESULTS_DIR, exist_ok=True)
INERT = ["TPO", "Megalin", "PSPN", "EPO", "EGF"]
#INERT = ["Noggin", "Decorin", "PSPN", "GDNF", "Megalin", "PRL", "EPO", "TPO", "FGF-beta", "EGF", "HGF", "VEGF", "IGF-1"]
QUERY_DONORS = ["Donor1", "Donor4", "Donor9", "Donor12"]

## 1. Run Rhaister and save predictions (data are downloaded from HuggingFace on first use)

In [2]:
pred_dir = os.path.join(RH_DIR, "predictions")
safe = SPLIT.replace("/", "_")
F_TEST = os.path.join(pred_dir, f"{EXP}__{safe}_test.parquet")
if not os.path.exists(F_TEST):
    alt = os.path.join(pred_dir, f"{EXP}__{safe}.parquet")          # name used by HP_SAVE_PREDICTIONS=1
    if not os.path.exists(alt):
        env = dict(os.environ, HP_SAVE_PREDICTIONS="1", WANDB_MODE="disabled", HF_HUB_DISABLE_PROGRESS_BARS="1")
        r = subprocess.run(["python", "-m", "rhaister.train", EXP, "--split", SPLIT], cwd=RH_DIR, env=env,
                           capture_output=True, text=True)
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
    F_TEST = alt
print("test predictions:", F_TEST, os.path.exists(F_TEST))

# training-row truths from Rhaister's own cache
import sys, json; sys.path.insert(0, RH_DIR)
from rhaister.prepare_combined import _cache_path
CACHE = _cache_path(*SPLIT.split("/", 1))
print("Rhaister cache:", CACHE, os.path.exists(os.path.join(CACHE, "D_train.npy")))

test predictions: Rhaister/predictions/rescoring__parse_fewshot_donor_split_0_test.parquet True


/home/w202gxg/miniconda/envs/renv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Rhaister cache: /tmp/tahoe_cache_combined/parse/fewshot_donor_split_0 True


## 2. Load truths and predictions

In [3]:
def load(path, genes):
    df = pd.read_parquet(path, columns=["cell_line", "treatment", "gene", "d_true", "d_pred"])
    df["gene"] = df["gene"].astype(str)
    out = {}
    for (c, t), g in df.groupby(["cell_line", "treatment"], observed=True, sort=False):
        g = g.set_index("gene").reindex(genes)       # align genes to the cache order
        out[(str(c), str(t))] = (g.d_true.to_numpy(np.float64), g.d_pred.to_numpy(np.float64))
    return out, genes

meta_c = json.load(open(os.path.join(CACHE, "meta.json")))
D_train = np.load(os.path.join(CACHE, "D_train.npy"), mmap_mode="r")
GENES = pd.Index([str(g) for g in meta_c["gene_cols"]])          # column order of the cached matrices
TEST, _ = load(F_TEST, GENES)                                     # predictions reindexed to the same order
TRAIN = {(str(c), str(t)): (np.asarray(D_train[i], np.float64), None)
         for i, (c, t) in enumerate(zip(meta_c["train_cells"], meta_c["train_treatments"]))}
def donor_of(ctx):
    m = re.search(r"Donor\s?\d+", ctx); return m.group(0).replace(" ", "") if m else ctx.split("__")[0]
def celltype_of(ctx):
    return re.sub(r"_*Donor\s?\d+_*", "", ctx).strip("_ ")
print(f"test rows: {len(TEST)}, train rows: {len(TRAIN)}, genes: {len(GENES)}")
print("example contexts:", sorted({k[0] for k in TEST})[:4], "->", donor_of(next(iter(TEST))[0]), "/", celltype_of(next(iter(TEST))[0]))

test rows: 4452, train rows: 14735, genes: 2000
example contexts: ['Donor12__B_Intermediate_Memory', 'Donor12__B_Naive', 'Donor12__CD14_Mono', 'Donor12__CD16_Mono'] -> Donor12 / B_Intermediate_Memory


## 3. Predictors

In [4]:
TRUE = {k: v[0] for k, v in TRAIN.items()}; TRUE.update({k: v[0] for k, v in TEST.items()})
test_ctx = sorted({c for c, _ in TEST}); test_tr = sorted({t for _, t in TEST})
panel = sorted({t for c, t in TRAIN if c in test_ctx})
train_ctx = sorted({c for c, _ in TRAIN if c not in test_ctx})
by_ct = {}
for c in train_ctx: by_ct.setdefault(celltype_of(c), []).append(c)

def train_mean(ct, t):
    v = [TRUE[(c, t)] for c in by_ct.get(ct, []) if (c, t) in TRUE]
    return np.nanmean(v, 0) if v else None

PRED = {"Rhaister": {}, "additive": {}, "context mean": {}, "inert shift": {}}
for ctx in test_ctx:
    ct = celltype_of(ctx)
    obs_panel = {p: TRUE[(ctx, p)] for p in panel if (ctx, p) in TRUE and train_mean(ct, p) is not None}
    if len(obs_panel) < 5: continue
    ctx_mean = np.nanmean(list(obs_panel.values()), 0)
    offset = ctx_mean - np.nanmean([train_mean(ct, p) for p in obs_panel], 0)
    inert_tm = [train_mean(ct, k) for k in INERT]
    inert_tm = [v for v in inert_tm if v is not None]
    shift = np.nanmean(inert_tm, 0) if inert_tm else None
    for t in test_tr:
        if (ctx, t) not in TEST: continue
        tm = train_mean(ct, t)
        PRED["Rhaister"][(ctx, t)] = TEST[(ctx, t)][1]
        if tm is not None: PRED["additive"][(ctx, t)] = tm + offset
        PRED["context mean"][(ctx, t)] = ctx_mean
        if shift is not None: PRED["inert shift"][(ctx, t)] = shift
print({k: len(v) for k, v in PRED.items()})

{'Rhaister': 4452, 'additive': 4452, 'context mean': 4452, 'inert shift': 4452}


## 4. Scores: as published (PBS) and after inert re-referencing

In [5]:
def pearson(a, b):
    m = ~(np.isnan(a) | np.isnan(b)); a, b = a[m] - a[m].mean(), b[m] - b[m].mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

inert_test = [t for t in test_tr if t in INERT]
print("inert test cytokines used for re-referencing:", inert_test)
rows = []
for name, P in PRED.items():
    for ctx in test_ctx:
        keys = [(ctx, t) for t in test_tr if (ctx, t) in P and (ctx, t) in TEST]
        ik = [k for k in keys if k[1] in INERT]
        if len(ik) < 3: continue
        ref_true = np.nanmean([TEST[k][0] for k in ik], 0)
        ref_pred = np.nanmean([P[k] for k in ik], 0)
        for k in keys:
            if k[1] in INERT: continue
            rows.append(dict(model=name, context=ctx, cytokine=k[1],
                             pbs=pearson(P[k], TEST[k][0]),
                             inert=pearson(P[k] - ref_pred, TEST[k][0] - ref_true)))
SC = pd.DataFrame(rows)
SC.to_csv(f"{RESULTS_DIR}/scores_per_context.csv", index=False)
SUMMARY = SC.groupby("model")[["pbs", "inert"]].mean().round(3)
SUMMARY["rank_pbs"] = SUMMARY.pbs.rank(ascending=False).astype(int)
SUMMARY["rank_inert"] = SUMMARY.inert.rank(ascending=False).astype(int)
SUMMARY = SUMMARY.sort_values("pbs", ascending=False)
SUMMARY.to_csv(f"{RESULTS_DIR}/scores_summary.csv")
SUMMARY

inert test cytokines used for re-referencing: ['EGF', 'EPO', 'PSPN']


,pbs,inert,rank_pbs,rank_inert
model,,,,
additive,0.476,0.349,1,1
Rhaister,0.461,0.308,2,2
context mean,0.303,-0.000,3,3
inert shift,0.035,-0.000,4,3


## How to read
- `pbs`: the published evaluation (Pearson of deltas against PBS), on immune test cytokines.
- `inert`: the same after removing, on both sides, the mean over inert test cytokines of the same context.
- If the gap between Rhaister and the trivial predictors (context mean, inert shift) grows under `inert`, part of the published scores came from the shared PBS shift; if ranks change, conclusions about which method is better depend on the evaluation.
- Note: Rhaister also predicts the inert cytokines, so its own inert predictions are used for its re-referencing; the simple predictors use theirs.